# 14. EarlyStopping и финальный test

Видео: [14. EarlyStopping / тест](https://www.youtube.com/watch?v=ctFMYYVj6ZA&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=15)

## Зачем этот урок

Обучение можно **не дожимать** все эпохи:
- val перестал улучшаться (плато) или начал расти (переобучение) → **стоп**;
- берём **лучшие** сохранённые веса (видео 13);
- на **test** смотрим **один раз**, одну выбранную модель.

> **Конспект:** [`d.4.3`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.3 Регуляризация в Deep Learning/d.4.3 Регуляризация в Deep Learning.md) **§2.4** (early stopping). Карта — [`0_как_учить_pytorch.md`](../0_как_учить_pytorch.md).

| блок | тема |
|:---|:---|
| 1 | простой счётчик `patience` (как у автора сначала) |
| 2 | класс `EarlyStop` (упрощённый `__call__`) |
| 3 | правило test: один раз, одна модель |
| 4 | `sorted` классов (ловушка классификации) |

**Как смотреть.** ~16 мин — целиком или: счётчик → test-правило → класс (по желанию).

**Как решать.** Замени `...` → проверка. Решения — `14_early_stopping_решения.ipynb`.

In [1]:
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)

CKPT = Path("_checkpoints_14")
CKPT.mkdir(exist_ok=True)


def need(cond, msg):
    assert cond, msg


class TinyReg(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(1, 1)

    def forward(self, x):
        return self.lin(x)


# синтетика (бутафория под цикл; не тема урока)
X = torch.linspace(-1, 1, 120).unsqueeze(1)
y = 2 * X + 1 + 0.08 * torch.randn_like(X)
n_tr, n_va = 70, 25
Xtr, ytr = X[:n_tr], y[:n_tr]
Xva, yva = X[n_tr : n_tr + n_va], y[n_tr : n_tr + n_va]
Xte, yte = X[n_tr + n_va :], y[n_tr + n_va :]
tr_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=16, shuffle=True)

print("torch", torch.__version__, "| train/val/test:", len(Xtr), len(Xva), len(Xte))

torch 2.14.1+cpu | train/val/test: 70 25 25


## 1. <span style="color:#7CDBA0">Счётчик patience</span> — простой Early Stopping

**Зачем:** если val `patience` эпох **не улучшался** → `break`.

```python
bad = 0
best_val = float("inf")
for epoch in range(TOTAL):
    # train ... val_loss ...
    if val_loss < best_val:
        best_val = val_loss
        bad = 0
        torch.save(model.state_dict(), "best.pt")
    else:
        bad += 1
        if bad >= patience:
            break
```

Связь с видео 13: при улучшении — save; счётчик сбрасываем. Порог «значимости» — как у Plateau; здесь достаточно `<`.

In [2]:
# 1.1 early stop простым счётчиком
torch.manual_seed(0)
model = TinyReg()
opt = torch.optim.Adam(model.parameters(), lr=0.08)
crit = nn.MSELoss()

TOTAL = 40
patience = 5
bad = 0
best_val = float("inf")
best_path = CKPT / "best_es.pt"
hist_val = []
stopped_epoch = -1

for epoch in range(TOTAL):
    model.train()
    for xb, yb in tr_loader:
        loss = crit(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        val_loss = crit(model(Xva), yva).item()
    hist_val.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        bad = 0
        torch.save(model.state_dict(), best_path)
        print(f"  ↑ epoch {epoch+1} val={val_loss:.4f}")
    else:
        bad += 1
        if bad >= patience:
            stopped_epoch = epoch
            print(f"stop at epoch {epoch+1} (patience={patience})")
            break

print("stopped_epoch:", stopped_epoch, "len hist:", len(hist_val), "best:", best_val)

  ↑ epoch 1 val=1.9369
  ↑ epoch 2 val=1.5495
  ↑ epoch 3 val=0.9471
  ↑ epoch 4 val=0.4601
  ↑ epoch 5 val=0.1981
  ↑ epoch 6 val=0.0861
  ↑ epoch 7 val=0.0315
  ↑ epoch 8 val=0.0070
stop at epoch 13 (patience=5)
stopped_epoch: 12 len hist: 13 best: 0.007027516141533852


In [3]:
need(best_path.is_file(), "сохраняй best при улучшении")
need(len(hist_val) < TOTAL, "должен сработать early stop до TOTAL")
need(stopped_epoch >= 0, "запомни эпоху остановки в stopped_epoch")
need(bad >= patience or stopped_epoch >= 0, "patience")
print("1.1 — верно")

1.1 — верно


## 2. <span style="color:#7CDBA0">Класс EarlyStop</span> (упрощённо)

У автора — класс с порогами как у Plateau. **Нам хватит:** `patience` + mode `"min"` для loss.

```python
es = EarlyStop(patience=5)
# после val:
if es(val_loss):   # True → пора остановиться
    break
```

`__call__` возвращает `True`, когда пора стоп.

In [4]:
# 2.1 класс EarlyStop
class EarlyStop:
    def __init__(self, patience=5):
        self.patience = patience
        self.bad = 0
        self.best = None

    def __call__(self, value: float) -> bool:
        """True = остановить обучение."""
        value = float(value)
        if self.best is None or value < self.best:
            self.best = value
            self.bad = 0
            return False
        self.bad += 1
        return self.bad >= self.patience


# быстрая проверка логики без обучения
es = EarlyStop(patience=2)
need(es(1.0) is False, "первая эпоха — запомнить best, не стоп")
need(es(0.9) is False, "улучшение — сброс bad")
need(es(0.95) is False, "1-е ухудшение — ещё не стоп")
need(es(0.96) is True, "2-е ухудшение подряд при patience=2 → стоп")
print("2.1 — верно")

2.1 — верно


## 3. <span style="color:#7CDBA0">Test один раз</span>

**Правило автора (и практики):**
1. Выбрал **одну** модель (обычно best по val).
2. Посчитал метрику на **test**.
3. **Больше не крутишь** гиперпараметры / не перебираешь другие checkpoint’ы под test.

Иначе подгоняешь решение под test (не модель «запомнила» test, а **ты** выбираешь по цифрам test) — оценка перестаёт быть честной.

Если test не устроил → новый эксперимент (заново модель/обучение; при возможности — новое разбиение). Не «подобрать другую сохранёнку, пока test не понравится».

In [5]:
# 3.1 один прогон test на best весах
model_t = TinyReg()
model_t.load_state_dict(torch.load(best_path, map_location="cpu", weights_only=True))
model_t.eval()
with torch.no_grad():
    test_loss = crit(model_t(Xte), yte).item()

print("test_loss:", round(test_loss, 4), "| best_val:", round(best_val, 4))
# для урока просто посчитай; «один раз» — правило процесса, не assert
need(isinstance(test_loss, float), "test_loss = ... .item()")
print("3.1 — верно")

test_loss: 0.0094 | best_val: 0.007
3.1 — верно


## 4. <span style="color:#7CDBA0">sorted + class_to_idx</span> — важно не забыть

Две связанные привычки для **классификации** (видео 14):

### 1) Список папок классов — всегда `sorted`

`os.listdir` **без** сортировки может отдать папки в разном порядке на разных ОС / после пересоздания папок.

Тогда `class_to_idx` другой: при обучении «футболка → 0», при load/test «футболка → 5». Веса те же, а метки съехали → accuracy ≈ случайная (**~10% на 10 классах**), loss огромный. Кажется, что модель «разучилась» — на деле нумерация классов другая.

```python
self.classes = sorted(...)  # как в ImageDataset проекта
self.class_to_idx = {name: i for i, name in enumerate(self.classes)}
```

### 2) Сохранять `class_to_idx` в checkpoint

```python
torch.save({
    "model": model.state_dict(),
    "class_to_idx": train_data.class_to_idx,  # расшифровка: pred=3 → какое имя
    ...
}, "ckpt.pt")
```

**Зачем**, если «всё равно заново сделаю в том же порядке»:
- порядок без `sorted` **не гарантирован**;
- через месяц / на другом ПК открыл только `.pt` — откуда знать, что значит класс `3`?
- в проде рядом с весами нужна легенда меток.

Регрессия (квадраты) — этого словаря нет. Классификация с папками по классам — **и sorted, и save `class_to_idx`**.

In [6]:
# 4.1 напоминание
classes = ["tshirt", "bag", "shoe"]
# плохо: list(os.listdir(...)) без sort
# хорошо:
idx = {name: i for i, name in enumerate(sorted(classes))}
need(idx["bag"] == 0 and idx["shoe"] == 1 and idx["tshirt"] == 2, "алфавитный порядок")
print("4.1 — верно")

4.1 — верно


## Итог

| идея | суть |
|:---|:---|
| Early stopping | `patience` эпох без улучшения val → stop |
| + video 13 | при улучшении save best; в конце load best |
| Test | **один раз**, одна модель (не подгонять под test) |
| `sorted` + `class_to_idx` | стабильные метки; сохранить словарь с весами |

Дальше: раздел **«Видео 14»** в `project/fashion_mnist.ipynb`. Потом — Dropout (видео 15).